# Fetch Laser Execution Data

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")
print(f"Python version: {sys.version}")

In [ ]:
from femtum_sdk.api.contracts.v1.storage.health_pb2 import HealthDto, HealthRequest
from femtum_sdk.api import DatastoreApi

api = DatastoreApi(hostUrl="localhost:5208")
api.__enter__()
result: HealthDto = api.health.GetHealth(HealthRequest())
print(result)


## Fetch Laser Executions Data

In [ ]:
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2 import LaserExecutionPage, ListByPageLaserExecutionsRequest
from femtum_sdk.api.contracts.v1.storage.laser_execution_pb2_grpc import (
    LaserExecutionServiceStub,
)

laser_execution_service = LaserExecutionServiceStub(api.grpc_channel)
page: LaserExecutionPage = api.laser_execution.ListByPage(ListByPageLaserExecutionsRequest(
    PageSize=10,
    PageNumber=0,
))

print(page)


## Fetch Images Attached to Execution

Fetch image data

In [ ]:
from femtum_sdk.api.contracts.file_pb2 import Attachment

itemIdToFind = "7"
laser_execution = next(item for item in page.Items if item.Id == itemIdToFind)

print(laser_execution.Attachments)
attachment_ids = [attachment.Id for attachment in laser_execution.Attachments]

attachments: list[Attachment] = []
for attachment_id in attachment_ids:
    attachment = api.attachment.Download(id=attachment_id)
    attachments.append(attachment)

Display images

In [ ]:
from IPython.display import display, Image, Markdown

for attachment in attachments:
    if attachment.MimeType.startswith("image/"):
        display(Markdown(f"**{attachment.Name}**"))
        display(Image(data=attachment.BlobData, format=attachment.MimeType.split("/")[-1]))
    else:
        print(f"Skipping non-image attachment: {attachment.Name} ({attachment.MimeType})")

## Cleanup

In [ ]:
api.__exit__(None, None, None)
